# HyWindSea SANTANDER — WRF day selection

Which days to simulate with WRF, chosen from the full ERA5 record: daily winds →
PCA → MDA.

**Reads** `inputs/wind_era5_full.nc`
**Writes** `outputs/SANTANDER/times/times_wrf_santander.txt`

In [ ]:
import os

import numpy as np
import pandas as pd
import xarray as xr
from bluemath_tk.datamining.pca import PCA
from utils.operations import max_diss

In [ ]:
SITE = "SANTANDER"

POOL = ("1982", "2023")  # ERA5 goes back further; this is the pool that was used
N_PCS = 5  # 98.6 % of the variance
N_DAYS = 365  # days to simulate

TIMES_DIR = f"outputs/{SITE}/times"
os.makedirs(TIMES_DIR, exist_ok=True)

## Daily winds

The selection works on daily fields: one wind map per day instead of 24, which is
the scale WRF was run at.

In [ ]:
full = xr.open_dataset("inputs/wind_era5_full.nc")
daily = full.sel(time=slice(*POOL)).resample(time="1D").mean()

print(
    f"{daily.sizes['time']:,} days x "
    f"{daily.sizes['lat'] * daily.sizes['lon']} ERA5 nodes"
)

15,340 days x 20 ERA5 nodes


## PCA

Each daily map is 20 nodes × 2 components = 40 numbers. The PCA turns that into
five, which is what makes the selection tractable.

In [ ]:
pca = PCA(n_components=N_PCS)

pca.fit_transform(
    data=daily[["u10", "v10"]],
    vars_to_stack=["u10", "v10"],
    coords_to_stack=["lat", "lon"],
    pca_dim_for_rows="time",
    # The default, but load-bearing: without standardising, the components come
    # out different and the MDA below picks other days.
    scale_data=True,
)

days = pd.DatetimeIndex(pca.pcs_df.index).normalize()

print(
    f"{pca.pcs_df.shape[1]} PCs · "
    f"{pca.cumulative_explained_variance_ratio[-1]:.1%} of the variance"
)

2026-08-27 10:47:42,219 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-08-27 10:47:42,221 - PCA - WARNING - Using 20 out of 20 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method


5 PCs · 98.6% of the variance


## MDA

Maximum Dissimilarity picks the day furthest from everything already picked, over
and over. The result covers the edges of the cloud — the unusual days — rather
than the crowded middle, which is what a training set needs.

`max_diss` is the teslakit algorithm the original selection used. `MDA` from
`bluemath_tk` is the same idea with two different defaults and returns a
different set of days.

In [ ]:
order = max_diss(pca.pcs_df, num_centers=N_DAYS)
selection = days[order]

print(
    f"{len(selection)} days, {selection.min():%Y-%m-%d} to {selection.max():%Y-%m-%d}"
)
print(f"first: {', '.join(f'{d:%Y-%m-%d}' for d in selection[:5])} ...")

365 days, 1982-03-07 to 2023-11-04
first: 1987-10-15, 2001-11-14, 2018-01-16, 2011-03-16, 1994-08-10 ...


## Save

In [ ]:
np.savetxt(
    f"{TIMES_DIR}/times_wrf_santander.txt",
    selection.strftime("%Y-%m-%d").values,
    fmt="%s",
)

print(f"saved to {TIMES_DIR}/times_wrf_santander.txt")

saved to outputs/SANTANDER/times/times_wrf_santander.txt
